In [ ]:
%pip install gradio

In [8]:
import gradio as gr
import pandas as pd
import pickle
from src.data_preprocessing import encode_features

# Rutas de los modelos
models = {
    "Random Forest": "../models/random_forest.pkl",
    "Logistic Regression": "../models/logistic_regression.pkl",
    "XGBoost": "../models/xgboost.pkl"
}

# Cargar todos los modelos
loaded_models = {}
for name, path in models.items():
    with open(path, "rb") as f:
        loaded_models[name] = pickle.load(f)

# Ejemplo de cliente
example_customer = {
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "No",
    "tenure": 12,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No",
    "OnlineBackup": "Yes",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "Yes",
    "StreamingMovies": "Yes",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 75.65,
    "TotalCharges": 910.50
}

# Función de predicción para múltiples modelos
def predict_churn_multiple_models(
    gender, SeniorCitizen, Partner, Dependents, tenure, PhoneService,
    MultipleLines, InternetService, OnlineSecurity, OnlineBackup,
    DeviceProtection, TechSupport, StreamingTV, StreamingMovies,
    Contract, PaperlessBilling, PaymentMethod, MonthlyCharges, TotalCharges
):
    try:
        df = pd.DataFrame([{
            "customerID": "0000-0",  # columna ficticia
            "gender": gender,
            "SeniorCitizen": SeniorCitizen,
            "Partner": Partner,
            "Dependents": Dependents,
            "tenure": tenure,
            "PhoneService": PhoneService,
            "MultipleLines": MultipleLines,
            "InternetService": InternetService,
            "OnlineSecurity": OnlineSecurity,
            "OnlineBackup": OnlineBackup,
            "DeviceProtection": DeviceProtection,
            "TechSupport": TechSupport,
            "StreamingTV": StreamingTV,
            "StreamingMovies": StreamingMovies,
            "Contract": Contract,
            "PaperlessBilling": PaperlessBilling,
            "PaymentMethod": PaymentMethod,
            "MonthlyCharges": MonthlyCharges,
            "TotalCharges": TotalCharges
        }])
        
        df_encoded = encode_features(df, training=False)
        
        results = {}
        for name, model in loaded_models.items():
            pred = model.predict(df_encoded)[0]
            # Verifica si el modelo tiene predict_proba
            if hasattr(model, "predict_proba"):
                prob = model.predict_proba(df_encoded)[0][1]
                prob_text = f"{prob*100:.2f}%"
            else:
                prob_text = "N/A"
            status = "Churn ⚠️" if pred == 1 else "No Churn ✅"
            results[name] = f"{status}, Probabilidad de churn: {prob_text}"
        
        return results

    except Exception as e:
        return {"Error": str(e)}

# Inputs visuales
inputs = [
    gr.Dropdown(["Female", "Male"], label="Gender"),
    gr.Slider(0, 1, step=1, label="SeniorCitizen"),
    gr.Dropdown(["Yes", "No"], label="Partner"),
    gr.Dropdown(["Yes", "No"], label="Dependents"),
    gr.Slider(0, 72, step=1, label="Tenure"),
    gr.Dropdown(["Yes", "No"], label="PhoneService"),
    gr.Dropdown(["Yes", "No", "No phone service"], label="MultipleLines"),
    gr.Dropdown(["DSL", "Fiber optic", "No"], label="InternetService"),
    gr.Dropdown(["Yes", "No", "No internet service"], label="OnlineSecurity"),
    gr.Dropdown(["Yes", "No", "No internet service"], label="OnlineBackup"),
    gr.Dropdown(["Yes", "No", "No internet service"], label="DeviceProtection"),
    gr.Dropdown(["Yes", "No", "No internet service"], label="TechSupport"),
    gr.Dropdown(["Yes", "No", "No internet service"], label="StreamingTV"),
    gr.Dropdown(["Yes", "No", "No internet service"], label="StreamingMovies"),
    gr.Dropdown(["Month-to-month", "One year", "Two year"], label="Contract"),
    gr.Dropdown(["Yes", "No"], label="PaperlessBilling"),
    gr.Dropdown([
        "Electronic check", "Mailed check",
        "Bank transfer (automatic)", "Credit card (automatic)"
    ], label="PaymentMethod"),
    gr.Number(label="MonthlyCharges", value=example_customer["MonthlyCharges"]),
    gr.Number(label="TotalCharges", value=example_customer["TotalCharges"])
]

# Ejemplo clicable
examples = [[
    example_customer["gender"],
    example_customer["SeniorCitizen"],
    example_customer["Partner"],
    example_customer["Dependents"],
    example_customer["tenure"],
    example_customer["PhoneService"],
    example_customer["MultipleLines"],
    example_customer["InternetService"],
    example_customer["OnlineSecurity"],
    example_customer["OnlineBackup"],
    example_customer["DeviceProtection"],
    example_customer["TechSupport"],
    example_customer["StreamingTV"],
    example_customer["StreamingMovies"],
    example_customer["Contract"],
    example_customer["PaperlessBilling"],
    example_customer["PaymentMethod"],
    example_customer["MonthlyCharges"],
    example_customer["TotalCharges"]
]]

# Crear interfaz Gradio
iface = gr.Interface(
    fn=predict_churn_multiple_models,
    inputs=inputs,
    outputs=gr.JSON(label="Resultados de todos los modelos"),
    examples=examples,
    title="Predicción de Churn con Múltiples Modelos",
    description="Compara la predicción de varios modelos de churn al mismo tiempo."
)

# Lanzar localmente
iface.launch(share=False)


* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.
